# Notebook 4: Data types, missing values and the P/E problem
*Kaggle Pandas lesson: "Data Types and Missing Values"*

**Module question: what makes a stock risky?** So far we have quietly stepped around the empty cells in
our table. Today they are the topic. In financial data a missing value is rarely random: no cost of
goods sold means a bank, no R&D means a company that does not innovate, no IPO date means an old firm.
Deciding what to do with a NaN is a modeling decision, and we will see cases where zero is right,
where it is debatable, and where the honest answer is "undefined".

## Learning goals
* read and change the **data type** of a column (`dtype`, `dtypes`, `astype`);
* find missing values (`isnull`, `notnull`) and count them;
* treat them deliberately: `fillna`, `replace`, and setting a subset of a column to NaN with `loc`.

## Setup

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/assacohen1/fin6040-pandas-data/main/"
firms = pd.read_csv(DATA_URL + "companies_2025.csv")

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 40)

In [ ]:
firms["leverage"] = firms.total_debt / (firms.total_debt + firms.market_cap)

## In class

### 1. Data types
Every column has one type: `float64` for numbers with decimals, `int64` for whole numbers, `object`
for text (and for mixtures). `dtype` gives one column's type, `dtypes` all of them.

In [ ]:
firms.market_cap.dtype

In [ ]:
firms.industry_code.dtype

In [ ]:
firms.ticker.dtype

In [ ]:
firms.dtypes

`astype` converts. Two finance examples: dropping the decimals of market cap, and turning the GICS
industry code into text. A code like 452020 is a *label*, not a quantity: its first two digits are the
sector, the first four the industry group. As text we can slice it.

In [ ]:
firms.market_cap.astype(int)

In [ ]:
firms.industry_code.astype(str)

`firms.employees.astype(int)` would **fail**: the column has missing values, and NaN is a float that
has no integer version. Fill or drop the NaNs first.

The index has a type too:

In [ ]:
firms.index.dtype

In [ ]:
firms.set_index("ticker").index.dtype

### 2. Finding missing values
`isnull()` gives True where a value is missing. Summing it counts the missing values; on the whole table
this is the most useful data check in pandas:

In [ ]:
firms.isnull().sum()

Missing is not random. Which sectors have no cost of goods sold?

In [ ]:
firms[pd.isnull(firms.cogs)].sector.value_counts()

And who reports R&D?

In [ ]:
firms[firms.r_and_d.notnull()].sector.value_counts()

### 3. Replacing missing values: `fillna`
`fillna(value)` returns a copy with NaN replaced. Three typical choices, each a judgment call:
* no reported dividend means no dividend paid: fill with **0** (correct);
* an unknown number of employees: fill with the **median** (common, crude);
* an unknown state: fill with a **label** such as "Unknown" (fine for text; never do this on a numeric column, it turns the column into text).

Like every pandas method, `fillna` does not change `firms` unless you assign the result back.

In [ ]:
firms.dividends.fillna(0)

In [ ]:
firms.employees.fillna(firms.employees.median())

In [ ]:
firms.state.fillna("Unknown")

In [ ]:
firms["dividends"] = firms.dividends.fillna(0)     # assign back to keep it
firms.dividends.isnull().sum()

### 4. Replacing values that are present: `replace`
`replace(old, new)` swaps one value for another; a dictionary swaps several at once. Handy for short labels.

In [ ]:
firms.sector.replace("Information Technology", "IT")

In [ ]:
short = {"Information Technology": "IT", "Consumer Discretionary": "Cons. Disc.",
         "Consumer Staples": "Staples", "Communication Services": "Comm. Svcs.",
         "Health Care": "Health"}
firms["sector_short"] = firms.sector.replace(short)
firms.sector_short.value_counts()

**NaN is not equal to anything, not even to itself.** `float("nan") == float("nan")` is `False`, and
`NaN > 0` is `False` too. Never test for a missing value with `==`; use `isnull()` / `notnull()`.

In [ ]:
float("nan") == float("nan")

### 5. Setting part of a column to NaN (the P/E problem, solved properly)
Notebook 2 computed P/E with a row-by-row function. The idiomatic way is two lines: divide, then set the
meaningless cases to NaN with `loc[condition, column] = value`. This is also the correct way to change
a subset of a column in general (assigning to a filtered copy, `firms[cond]["pe"] = ...`, does nothing).

In [ ]:
firms["pe"] = firms.market_cap / firms.net_income
firms.loc[firms.net_income <= 0, "pe"] = float("nan")
firms.pe.describe()

Compare with the naive version: negative P/Es, a meaningless mean, and a minimum that is not 'cheap' but nonsense.

In [ ]:
(firms.market_cap / firms.net_income).describe()

## Exercises

### Exercise 1: Types and codes

(a) Assign the data type of the `market_cap` column to `mc_dtype`.
(b) Create `code_str`, the `industry_code` column converted to strings, and `industry_group`, the first
four characters of each code (`map` with a `lambda` from Notebook 2). How many distinct industry groups are there?

<details><summary>Hint</summary>

`.dtype`; `.astype(str)`; `code_str.map(lambda s: s[:4])`.

</details>

In [ ]:
mc_dtype = ____
code_str = ____
industry_group = ____

print(mc_dtype)
print(len(industry_group.unique()), "industry groups")

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert mc_dtype == "float64"
assert code_str.dtype == object
assert len(code_str.iloc[0]) == 6
assert len(industry_group.unique()) == 25
print("Looks right!")

### Exercise 2: Missing R&D

How many firms have no `r_and_d` value? Assign the count to `n_missing_rd`. Then show, with `value_counts`, which sectors these firms belong to.

<details><summary>Hint</summary>

`isnull().sum()`.

</details>

In [ ]:
n_missing_rd = ____

print(n_missing_rd)
firms[firms.r_and_d.isnull()].sector.value_counts()

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert n_missing_rd == 921
print("Looks right!")

### Exercise 3: Home states

Create `state_counts`: the number of firms per `state`, with missing states labelled `"Unknown"`, most common state first.

<details><summary>Hint</summary>

`fillna("Unknown")` first, then `value_counts()`.

</details>

In [ ]:
state_counts = ____

state_counts.head(12)

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert state_counts.sum() == len(firms)
assert state_counts.index[0] == 'CA'
assert state_counts["Unknown"] == 178
print("Looks right!")

### Exercise 4: Dividend payers

`dividends` was already filled with 0 above. Create a True/False column `pays_dividend` (dividends
greater than 0). Then assign the mean `vol_2025` of the payers to `payer_vol` and of the non-payers to
`nonpayer_vol`. Which group is riskier?

<details><summary>Hint</summary>

`~` flips a True/False Series: `firms.loc[~firms.pays_dividend, "vol_2025"]` are the non-payers.

</details>

In [ ]:
firms["pays_dividend"] = ____
payer_vol = ____
nonpayer_vol = ____

print(f"payers {payer_vol:.3f}   non-payers {nonpayer_vol:.3f}")

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert firms.dividends.isnull().sum() == 0
assert int(firms.pays_dividend.sum()) == 1165
assert round(payer_vol, 4) == 0.3433
assert payer_vol < nonpayer_vol
print("Looks right!")

### Exercise 5: P/E done right

The column `pe` was built in class. Assign the median P/E to `median_pe`, and the number of firms without a P/E to `n_no_pe`.

<details><summary>Hint</summary>

`median()` skips NaN by itself; `isnull().sum()` counts them.

</details>

In [ ]:
median_pe = ____
n_no_pe = ____

print(f"median P/E {median_pe:.1f}; {n_no_pe} firms have no P/E")

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert n_no_pe == (firms.net_income <= 0).sum()
assert firms.pe.min() > 0
assert round(median_pe, 2) == 22.3
print("Looks right!")

### Exercise 6: Firm age from a date string

Create `firms["ipo_year"]` from `ipo_date`: convert the column to strings, keep the first 4 characters
(`map` with a `lambda`), and convert the result to `float` (the missing dates become NaN by themselves).
Then assign three averages of `vol_2025`: `young_vol` for firms with `ipo_year` of 2020 or later,
`old_vol` for firms listed before 2020, and `unknown_vol` for firms with no IPO date.

<details><summary>Hint</summary>

`firms.ipo_date.astype(str).map(lambda d: d[:4]).astype(float)`; then three `loc[condition, "vol_2025"].mean()` calls with `>=`, `<` and `isnull()`.

</details>

In [ ]:
firms["ipo_year"] = ____
young_vol = ____
old_vol = ____
unknown_vol = ____

print(f"listed 2020 or later {young_vol:.3f}   earlier {old_vol:.3f}   unknown {unknown_vol:.3f}")

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert firms.ipo_year.isnull().sum() == firms.ipo_date.isnull().sum()
assert round(young_vol, 4) == 0.6826
assert young_vol > old_vol
print("Looks right!")

## Finance insight

Missing values in financial data carry information: no COGS means a bank, no R&D means a firm that does
not innovate, no IPO date in Compustat means a company listed long ago. And those old, dividend-paying,
profitable firms are the calm ones: dividend payers average a volatility of 0.34 against
0.61 for non-payers; firms listed in 2020 or later average 0.68
against 0.43 for older listings (and 0.42 for the oldest, whose IPO
date is unknown). **Age and payout join sector, profitability and size on the list of risk drivers.**

How you treat a NaN is a modeling decision. Zero is right for dividends. The median is a crude patch for
employees. For the P/E of a loss-maker the honest answer is *undefined*: a negative P/E is never "cheap",
and any average or ranking that includes it is wrong. Set it to NaN and let `median`, `describe` and
`groupby` skip it, which they do by default.